In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import re, os
from lxml import etree

MODELOS = '/content/drive/MyDrive/TESIS_CA/Pipeline/Intentos/MODELOS_1_v2'
SITIOS = ['Las_Docas', 'Algarrobo', 'Navidad', 'Topocalma', 'Ilque', 'San_Antonio', 'Pargua', 'Los_Chonos']
ETAPAS = ['sbml', 'sbml_curado', 'sbml_curado_final', 'sbml_curado_final_v2']

def buscar_sbml(carpeta):
    for f in os.listdir(carpeta):
        if f.endswith('.sbml'):
            return os.path.join(carpeta, f)
    raise FileNotFoundError(carpeta)

def contar_reacciones_genes(sbml_path):
    n_reacciones = 0
    genes = set()
    for event, elem in etree.iterparse(sbml_path, events=("end",), huge_tree=True):
        if etree.QName(elem).localname == "reaction":
            n_reacciones += 1
            texto = "".join(elem.itertext())
            m = re.search(r'GENE_ASSOCIATION:\s*(.*)', texto)
            if m:
                tokens = re.findall(r'[\w\.\-]+', m.group(1))
                genes.update(t for t in tokens if t.lower() not in ('and', 'or'))
            elem.clear()
    return n_reacciones, genes

resultados = {}  # resultados[sitio][etapa] = {'reacciones':..., 'genes': set(...)}

for sitio in SITIOS:
    resultados[sitio] = {}
    for etapa in ETAPAS:
        carpeta = os.path.join(MODELOS, sitio, etapa)
        try:
            ruta = buscar_sbml(carpeta)
            n_rxn, genes = contar_reacciones_genes(ruta)
            resultados[sitio][etapa] = {'reacciones': n_rxn, 'genes': genes}
        except FileNotFoundError:
            resultados[sitio][etapa] = None
            print(f"  [!] no se encontró .sbml para {sitio}/{etapa}")

# ---- tabla resumen ----
import pandas as pd
filas = []
for sitio in SITIOS:
    fila = {'sitio': sitio}
    for etapa in ETAPAS:
        r = resultados[sitio][etapa]
        fila[f'{etapa}_rxn']   = r['reacciones'] if r else None
        fila[f'{etapa}_genes'] = len(r['genes']) if r else None
    r_cur, r_fin = resultados[sitio]['sbml_curado'], resultados[sitio]['sbml_curado_final']
    fila['genes_perdidos_curado_a_final'] = (
        len(r_cur['genes'] - r_fin['genes']) if r_cur and r_fin else None
    )
    filas.append(fila)

df = pd.DataFrame(filas)
print(df.to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
      sitio  sbml_rxn  sbml_genes  sbml_curado_rxn  sbml_curado_genes  sbml_curado_final_rxn  sbml_curado_final_genes  sbml_curado_final_v2_rxn  sbml_curado_final_v2_genes  genes_perdidos_curado_a_final
  Las_Docas      5145      265782             6787             265782                   6781                   265782                      6775                      265782                              0
  Algarrobo      5696      444330             7487             444330                   7480                   444330                      7473                      444330                              0
    Navidad      4865      185084             6391             185084                   6385                   185084                      6378                      185084                              0
  Topocalma      5397      400278             7139         